In [1]:
import pandas as pd

df = pd.read_csv('data/Clean_Train.csv')

macro_mapping = {
    2: 0, 8: 0, 9: 0, 14: 0, 15: 0, 18: 0,
    0: 1, 3: 1, 4: 1, 10: 1, 17: 1,
    7: 2, 16: 2, 19: 2,
    1: 3, 6: 3, 12: 3, 13: 3,
    5: 4, 11: 4
}

df['Macro_Label'] = df['Label'].map(macro_mapping)
df = df.dropna(subset=['Macro_Label'])
df['Macro_Label'] = df['Macro_Label'].astype(int)

df.to_csv('data/Hierarchical_Train.csv', index=False)

print("Process Finished!")
print("Data Shape:", df.shape)
print("\nMacro Classes Distribution:")
print(df['Macro_Label'].value_counts())

Process Finished!
Data Shape: (47535, 4)

Macro Classes Distribution:
Macro_Label
0    15056
1    10907
3     9584
2     7577
4     4411
Name: count, dtype: int64


In [2]:
import os
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

print("Checking CUDA...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading hierarchical data...")
df = pd.read_csv('data/Hierarchical_Train.csv')
df = df.dropna(subset=['TEXT', 'Macro_Label'])

X = df['TEXT'].tolist()
y = df['Macro_Label'].tolist()

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Computing macro class weights...")
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Loading tokenizer & Tokenizing...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(X_train, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
test_encodings = tokenizer(X_test, padding="max_length", truncation=True, max_length=64, return_tensors="pt")

class MacroDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MacroDataset(train_encodings, y_train)
test_dataset = MacroDataset(test_encodings, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

print("Loading BERT for Macro Classification (5 Classes)...")
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

epochs = 3
total_steps = len(train_loader) * epochs
num_warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps)

loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

best_f1 = 0.0
print("Starting Macro BERT Training...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    step = 0
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fct(outputs.logits, labels)
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        step += 1
        
        if step % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Step {step}/{len(train_loader)} - Loss: {loss.item():.4f}")
            
    print(f"\n--> Epoch {epoch+1} Finished. Evaluating Macro Model...")
    model.eval()
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            
            val_preds.extend(batch_preds)
            val_targets.extend(labels.cpu().numpy())
            
    epoch_acc = accuracy_score(val_targets, val_preds)
    epoch_f1 = f1_score(val_targets, val_preds, average='weighted', zero_division=0)
    
    print(f"Epoch {epoch+1} Macro Results -> Accuracy: {epoch_acc:.4f} | F1-Score: {epoch_f1:.4f}")
    
    if epoch_f1 > best_f1:
        best_f1 = epoch_f1
        print(f" New Best Macro Model secured ({best_f1:.4f})! Saving...")
        os.makedirs('models/macro_bert_model', exist_ok=True)
        model.save_pretrained('models/macro_bert_model')
        tokenizer.save_pretrained('models/macro_bert_model')
    print("-" * 50)

print("Macro Model Training Done!")

c:\Users\user\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checking CUDA...
Using device: cuda
Loading hierarchical data...
Splitting data...
Computing macro class weights...
Loading tokenizer & Tokenizing...


Loading BERT for Macro Classification (5 Classes)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5101.59it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Starting Macro BERT Training...
Epoch 1/3 - Step 200/2377 - Loss: 1.5772
Epoch 1/3 - Step 400/2377 - Loss: 1.6023
Epoch 1/3 - Step 600/2377 - Loss: 1.1790
Epoch 1/3 - Step 800/2377 - Loss: 0.7539
Epoch 1/3 - Step 1000/2377 - Loss: 1.1426
Epoch 1/3 - Step 1200/2377 - Loss: 0.9162
Epoch 1/3 - Step 1400/2377 - Loss: 0.8949
Epoch 1/3 - Step 1600/2377 - Loss: 1.2394
Epoch 1/3 - Step 1800/2377 - Loss: 0.8245
Epoch 1/3 - Step 2000/2377 - Loss: 0.8772
Epoch 1/3 - Step 2200/2377 - Loss: 1.4103

--> Epoch 1 Finished. Evaluating Macro Model...
Epoch 1 Macro Results -> Accuracy: 0.5545 | F1-Score: 0.5534
 New Best Macro Model secured (0.5534)! Saving...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]


--------------------------------------------------
Epoch 2/3 - Step 200/2377 - Loss: 1.5507
Epoch 2/3 - Step 400/2377 - Loss: 1.1192
Epoch 2/3 - Step 600/2377 - Loss: 1.0303
Epoch 2/3 - Step 800/2377 - Loss: 0.8489
Epoch 2/3 - Step 1000/2377 - Loss: 1.0801
Epoch 2/3 - Step 1200/2377 - Loss: 0.9339
Epoch 2/3 - Step 1400/2377 - Loss: 0.6110
Epoch 2/3 - Step 1600/2377 - Loss: 1.2339
Epoch 2/3 - Step 1800/2377 - Loss: 0.9338
Epoch 2/3 - Step 2000/2377 - Loss: 0.7215
Epoch 2/3 - Step 2200/2377 - Loss: 0.6889

--> Epoch 2 Finished. Evaluating Macro Model...
Epoch 2 Macro Results -> Accuracy: 0.5641 | F1-Score: 0.5673
 New Best Macro Model secured (0.5673)! Saving...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.39it/s]


--------------------------------------------------
Epoch 3/3 - Step 200/2377 - Loss: 0.9716
Epoch 3/3 - Step 400/2377 - Loss: 0.7598
Epoch 3/3 - Step 600/2377 - Loss: 0.5423
Epoch 3/3 - Step 800/2377 - Loss: 0.6639
Epoch 3/3 - Step 1000/2377 - Loss: 0.6997
Epoch 3/3 - Step 1200/2377 - Loss: 0.6603
Epoch 3/3 - Step 1400/2377 - Loss: 1.0216
Epoch 3/3 - Step 1600/2377 - Loss: 0.4145
Epoch 3/3 - Step 1800/2377 - Loss: 0.6740
Epoch 3/3 - Step 2000/2377 - Loss: 0.8333
Epoch 3/3 - Step 2200/2377 - Loss: 0.5510

--> Epoch 3 Finished. Evaluating Macro Model...
Epoch 3 Macro Results -> Accuracy: 0.5658 | F1-Score: 0.5657
--------------------------------------------------
Macro Model Training Done!


In [3]:
import os
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('data/Hierarchical_Train.csv')
df = df.dropna(subset=['TEXT', 'Label', 'Macro_Label'])

os.makedirs('models', exist_ok=True)

macro_families = {
    0: "Hearts Family",
    1: "Joy & Laughs Family",
    2: "Energy & Cool Family",
    3: "Photos & Weather Family",
    4: "Events & Nations Family"
}

for macro_id, family_name in macro_families.items():
    print(f"\n--- Training Micro Model for: {family_name} (Macro {macro_id}) ---")
    
    sub_df = df[df['Macro_Label'] == macro_id]
    
    X = sub_df['TEXT'].tolist()
    y = sub_df['Label'].tolist()
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    sub_model = LogisticRegression(class_weight='balanced', max_iter=1000)
    sub_model.fit(X_train_vec, y_train)
    
    preds = sub_model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    print(f"{family_name} Micro Accuracy: {acc:.4f}")
    
    with open(f'models/micro_vectorizer_{macro_id}.pkl', 'wb') as f:
        pickle.dump(vectorizer, f)
    with open(f'models/micro_model_{macro_id}.pkl', 'wb') as f:
        pickle.dump(sub_model, f)

print("\nAll 5 Micro Models have been trained and secured successfully!")


--- Training Micro Model for: Hearts Family (Macro 0) ---
Hearts Family Micro Accuracy: 0.2865

--- Training Micro Model for: Joy & Laughs Family (Macro 1) ---
Joy & Laughs Family Micro Accuracy: 0.3483

--- Training Micro Model for: Energy & Cool Family (Macro 2) ---
Energy & Cool Family Micro Accuracy: 0.5891

--- Training Micro Model for: Photos & Weather Family (Macro 3) ---
Photos & Weather Family Micro Accuracy: 0.4752

--- Training Micro Model for: Events & Nations Family (Macro 4) ---
Events & Nations Family Micro Accuracy: 0.9117

All 5 Micro Models have been trained and secured successfully!
